In [ ]:
# Library
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer

# conect drive file
from google.colab import drive
drive.mount('/content/drive')

"""#**Adult Data**

# **Data Preprocessing**
"""

path = '/content/drive/MyDrive/adult.csv'
adult = pd.read_csv(path)
adult

# Find Missing Values ... (result: there is no missing values)
adult.isnull().sum()
# Handle missing values
adult.dropna(inplace=True)

# count ?
adult.isin(['?']).sum()

adult = adult.replace('?', np.nan)
adult.dropna(inplace=True)

(adult == '?').sum()

adult.isna().sum()

# label encoding for (income); convert  <=50K to 1 & >50K to 0
adult['income'] = adult['income'].map({'<=50K': 0, '>50K': 1})
adult

# Encode (one-hot encode) the categorical columns
# All categorical columns for encoding

categorical_col = ['workclass', 'education', 'marital-status', 'occupation','relationship', 'race', 'gender', 'native-country']

# Apply one-hot encoding
adult_encoded = pd.get_dummies(adult, columns=categorical_col, dtype=int)

# Display the result
display(adult_encoded.head())

#Separate features (X) and target (y)
X_adult = adult_encoded.drop('income', axis=1)
y_adult = adult_encoded['income']

# (( Split the data into training and testing sets "70% train, 30% test" ))
X_adult_test, X_adult_train, y_adult_test, y_adult_train = train_test_split(X_adult, y_adult, test_size=0.3, random_state=42)

# Standardize
scaler = StandardScaler()
# just select numerical column
num_of_col = ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
X_adult_train_scaled = scaler.fit_transform(X_adult_train[num_of_col])
X_adult_test_scaled = scaler.transform(X_adult_test[num_of_col])

from sklearn.svm import SVC
svm_linear_adult = SVC(kernel='linear', C=1, random_state=45, probability=True)
svm_linear_adult.fit(X_adult_train_scaled, y_adult_train)

#  Evaluate (Making predictions on the test set)
y_adult_pred_linear = svm_linear_adult.predict(X_adult_test_scaled)
# ((predictions)) vs (actual values)
print("Predicted values:", y_adult_pred_linear)
print()
print("Actual values:", y_adult_test.values)

# Compute metrics Linear SVM on Adult dataset
SVM_linear_accuracy = accuracy_score(y_adult_test, y_adult_pred_linear)
SVM_linear_precision = precision_score(y_adult_test, y_adult_pred_linear)
SVM_linear_recall = recall_score(y_adult_test, y_adult_pred_linear)
SVM_linear_f1 = f1_score(y_adult_test, y_adult_pred_linear)
SVM_linear_auc = roc_auc_score(y_adult_test, svm_linear_adult.decision_function(X_adult_test_scaled))

# Print the metrics
print("Accuracy:", SVM_linear_accuracy)
print("Precision:", SVM_linear_precision)
print("Recall:", SVM_linear_recall)
print("F1 Score:", SVM_linear_f1)
print("AUC:", SVM_linear_auc)

"""# **Model Building & Evaluation then Visualization**

*Logistic Regression model*
"""

# Logistic Regression model
LR_model = LogisticRegression(max_iter=2500)

# Train the model
LR_model.fit(X_adult_train_scaled, y_adult_train)

# Making predictions on the test set
y_adult_pred_LR = LR_model.predict(X_adult_test_scaled)

# ((predictions)) vs (actual values)
print("Predicted values:", y_adult_pred_LR)
print()
print("Actual values:", y_adult_test.values)

# Compute metrics Logistic Regression on Adult dataset
LR_accuracy_adult = accuracy_score(y_adult_test, y_adult_pred_LR) # accuracy

LR_precision_adult = precision_score(y_adult_test, y_adult_pred_LR) # precision

LR_recall_adult = recall_score(y_adult_test, y_adult_pred_LR) # recall

LR_f1_adult = f1_score(y_adult_test, y_adult_pred_LR) # f1

LR_roc_auc_adult = roc_auc_score(y_adult_test, LR_model.predict_proba(X_adult_test_scaled)[:, 1]) # ROC-AUC Score

print("Logistic Regression Evaluation on Adult Data:")
print(f"Accuracy: {LR_accuracy_adult:.4f}")
print(f"Precision: {LR_precision_adult:.4f}")
print(f"Recall: {LR_recall_adult:.4f}")
print(f"F1-Score: {LR_f1_adult:.4f}")
print(f"ROC-AUC Score: {LR_roc_auc_adult:.4f}")

# confusion matrices for Logistic Regression
LR_cm_adult = confusion_matrix(y_adult_test, y_adult_pred_LR)
print("/nconfusion matrices for Logistic Regression")
plt.figure(figsize=(4, 3))
sns.heatmap(LR_cm_adult, annot=True, fmt='d', cmap='Reds', cbar=False, xticklabels=['<=50K(0)', '>50K(1)'], yticklabels=['<=50K(0)', '>50K(1)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix for Logistic Regression (Adult Data)')
plt.show()

"""*Naive Bayes model*"""

# Naive Bayes model
nb_model = GaussianNB()

# Train the model
nb_model.fit(X_adult_train_scaled, y_adult_train)

# Making predictions on the test set
y_adult_pred_nb = nb_model.predict(X_adult_test_scaled)

# ((predictions)) vs (actual values)
print("Predicted values:", y_adult_pred_nb)
print()
print("Actual values:", y_adult_test.values)

# Compute metrics NB on Adult dataset
nb_accuracy = accuracy_score(y_adult_test, y_adult_pred_nb) # accuracy

nb_precision = precision_score(y_adult_test, y_adult_pred_nb) # precision

nb_recall = recall_score(y_adult_test, y_adult_pred_nb) # recall

nb_f1 = f1_score(y_adult_test, y_adult_pred_nb) # f1

nb_roc_auc = roc_auc_score(y_adult_test, nb_model.predict_proba(X_adult_test_scaled)[:, 1]) # ROC-AUC Score

print("\nNaive Bayes Evaluation on Adult Data:")
print(f"Accuracy: {nb_accuracy:.4f}")
print(f"Precision: {nb_precision:.4f}")
print(f"Recall: {nb_recall:.4f}")
print(f"F1-Score: {nb_f1:.4f}")
print(f"ROC-AUC Score: {nb_roc_auc:.4f}")

# confusion matrices for # Naive Bayes
nb_cm_adult = confusion_matrix(y_adult_test, y_adult_pred_nb)
print("/nconfusion matrices for Naive Bayes")
plt
plt.figure(figsize=(4, 3))
sns.heatmap(nb_cm_adult, annot=True, fmt='d', cmap='PuRd_r', cbar=False, xticklabels=['<=50K(0)', '>50K(1)'], yticklabels=['<=50K(0)', '>50K(1)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix for Naive Bayes (Adult Data)')
plt.show()

"""*BernoulliNB*"""

# BernoulliNB
b_model = BernoulliNB()

# Train the model
b_model.fit(X_adult_train_scaled, y_adult_train)

# # Making predictions on the test set
y_adult_pred_b = b_model.predict(X_adult_test_scaled)

# ((predictions)) vs (actual values)
print("Predicted values:", y_adult_pred_b)
print()
print("Actual values:", y_adult_test.values)

# Compute metrics BernoulliNB on Adult dataset
b_accuracy = accuracy_score(y_adult_test, y_adult_pred_b) # accuracy

b_precision = precision_score(y_adult_test, y_adult_pred_b) # precision

b_recall = recall_score(y_adult_test, y_adult_pred_b) # recall

b_f1 = f1_score(y_adult_test, y_adult_pred_b) # f1

b_roc_auc = roc_auc_score(y_adult_test, b_model.predict_proba(X_adult_test_scaled)[:, 1]) # ROC-AUC Score

print("\nBernoulli Naive Bayes Evaluation on Adult Data:")
print(f"Accuracy: {b_accuracy:.4f}")
print(f"Precision: {b_precision:.4f}")
print(f"Recall: {b_recall:.4f}")
print(f"F1-Score: {b_f1:.4f}")
print(f"ROC-AUC Score: {b_roc_auc:.4f}")

# confusion matrices for BernoulliNB
b_cm_adult = confusion_matrix(y_adult_test, y_adult_pred_b)
print("/nconfusion matrices for BernoulliNB")
plt.figure(figsize=(4, 3))
sns.heatmap(b_cm_adult, annot=True, fmt='d', cmap='YlOrRd', cbar=False, xticklabels=['<=50K(0)', '>50K(1)'], yticklabels=['<=50K(0)', '>50K(1)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix for BernoulliNB (Adult Data)')
plt.show()

# Get the predicted probabilities for the positive class for each model
# X_adult_test_scaled_full = scaler.transform(X_adult_test)

y_prob_LR = LR_model.predict_proba(X_adult_test_scaled)[:, 1]
y_prob_nb = nb_model.predict_proba(X_adult_test_scaled)[:, 1]
y_prob_b = b_model.predict_proba(X_adult_test_scaled)[:, 1]

# Calculate the False Positive Rate, True Positive Rate, and thresholds
fpr_log, tpr_log, _ = roc_curve(y_adult_test, y_prob_LR)
fpr_gnb, tpr_gnb, _ = roc_curve(y_adult_test, y_prob_nb)
fpr_bnb, tpr_bnb, _ = roc_curve(y_adult_test, y_prob_b)

# Calculate the Area Under the ROC Curve (AUC)
auc_log = roc_auc_score(y_adult_test, y_prob_LR)
auc_gnb = roc_auc_score(y_adult_test, y_prob_nb)
auc_bnb = roc_auc_score(y_adult_test, y_prob_b)

# plot the carve
plt.figure(figsize=(8, 6))
plt.plot(fpr_log, tpr_log, label=f"Logistic Regression (AUC = {auc_log:.3f})")
plt.plot(fpr_gnb, tpr_gnb, label=f"Gaussian NB (AUC = {auc_gnb:.3f})")
plt.plot(fpr_bnb, tpr_bnb, label=f"Bernoulli NB (AUC = {auc_bnb:.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Logistic Regression, Gaussian NB, Bernoulli NB")
plt.legend(loc="lower right")
plt.show()

"""# **Comparative Analysis**

Comparison of the performance: (Logistic Regression vs. Naive Bayes)
| Metric       | Logistic Regression | Gaussian Naive Bayes |  Birnoulli
|--------------|---------------------|----------------------|----------------
| Accuracy     | 0.81                | 0.8                  |0.80
| Precision    | 0.71                | 0.66                 |0.65
| Recall       | 0.37                | 0.3                  |0.36
| F1-Score     | 0.49                | 0.41                 |0.47
| ROC-AUC Score| 0.83                | 0.82                 |0.79

- Models performed lower than expectation which indicate that it's failed to identify the income '>$50K' (False Negative).
- Characteristics of dataset (feature types, distributions) influenced results because the dataset had both numbers and categories, Logistic Regression performed very well due to its suitability.
"""